# Tensor-network demo for a large non-Hermitian tight-binding model

This notebook mirrors `test.jl`, but keeps the calculation in small, inspectable steps. The workflow is:

1. load the tensor-network utilities,
2. encode position-dependent hopping amplitudes as diagonal MPOs with QTCI,
3. build the 3D tight-binding MPO,
4. add the imaginary onsite potential/loss term,
5. hermitize `omega * I - H`,
6. evaluate one real-space spectral point with the KPM recurrence.

For a quick local check, reduce `L` and `kpm_order` before running all cells.


In [25]:
using ITensors
using ITensorMPS
using QuanticsTCI
import TensorCrossInterpolation as TCI
using TCIITensorConversion
using Quantics

include("2D_lattice.jl")
include("NHtk.jl")
include("extra_util.jl")


extract_diagonal_to_mps (generic function with 1 method)

## Parameters

`L` sets the number of binary coordinates per spatial direction, so the large-system lattice has `2^L` sites along each axis. The hermitized Hamiltonian adds one auxiliary qubit.


In [26]:
const L = 4
const tx1 = 1.0
const tx2 = 1.4
const beta1 = 1 / 4
const beta2 = 1 / 4
const beta3 = 1 / 4
const phi = pi / 4
const lambda_strength = 5.0

const kx = 0.0
const ky = 4.7
const kpm_order = 100
const spectrum_scale = 12.0
const site_index = 0

const Nx = 2^L
const Ny = 2^L
const Nz = 2^L


16

## Helper MPOs for hopping and identity operators

The hopping amplitudes are classical functions over binary-encoded coordinates. QTCI compresses each function into an MPS, and `mps_to_diagonal_mpo` converts it into a diagonal MPO that can modulate shift operators.


In [27]:
function alternating_row_hopping(Ltot, tx_odd, tx_even, sites)
    xvals = range(0, 2^Ltot - 1; length=2^Ltot)

    function hopping_value(x)
        x_in_row = x % 2^L
        return iseven(x_in_row) ? tx_odd : tx_even
    end

    qtt, ranks, errors = quanticscrossinterpolate(Float64, hopping_value, xvals; tolerance=1e-8)
    tt = TCI.tensortrain(qtt.tci)
    hopping_mps = MPS(tt; sites)
    return mps_to_diagonal_mpo(hopping_mps, sites)
end

function staggered_column_hopping(Ltot, ty_odd, ty_even, sites)
    xvals = range(0, 2^Ltot - 1; length=2^Ltot)

    function hopping_value(x)
        x_in_row = x % 2^L
        row = div(x, 2^L)

        if iseven(row)
            return iseven(x_in_row) ? -ty_odd : ty_odd
        else
            return iseven(x_in_row) ? -ty_even : ty_even
        end
    end

    qtt, ranks, errors = quanticscrossinterpolate(Float64, hopping_value, xvals; tolerance=1e-8)
    tt = TCI.tensortrain(qtt.tci)
    hopping_mps = MPS(tt; sites)
    return mps_to_diagonal_mpo(hopping_mps, sites)
end

function identity_mpo(sites, nsites)
    ops = OpSum()
    for i in 1:nsites
        ops += 1 / nsites, "Id", i
    end
    return MPO(ComplexF64, ops, sites)
end

function vertical_hopping(Ltot, tz_odd, tz_even, sites)
    xvals = range(0, 2^Ltot - 1; length=2^Ltot)

    function hopping_value(x)
        z = div(x, 2^(2L))
        x_coord = mod(x, 2^L)
        y_coord = mod(div(x, 2^L), 2^L)

        sign_xy = iseven(x_coord + y_coord) ? 1.0 : -1.0
        hopping = iseven(z) ? tz_odd : tz_even
        return sign_xy * hopping
    end

    qtt, ranks, errors = quanticscrossinterpolate(Float64, hopping_value, xvals; tolerance=1e-8)
    tt = TCI.tensortrain(qtt.tci)
    hopping_mps = MPS(tt; sites)
    return mps_to_diagonal_mpo(hopping_mps, sites)
end


vertical_hopping (generic function with 1 method)

## Onsite potential

The onsite term is represented as the product of two diagonal MPOs: a slowly varying unit-cell envelope and the full coordinate envelope.


In [28]:
@inline function xyz_from_linear_index(i)
    i0 = i - 1
    x = mod(i0, Nx) + 1
    y = mod(div(i0, Nx), Ny) + 1
    z = div(i0, Nx * Ny) + 1
    return x, y, z
end

function onsite_envelope(i)
    x, y, z = xyz_from_linear_index(i)

    vx = cos(2pi * beta1 * (x - 1) + phi)
    vy = cos(2pi * beta2 * (y - 1) + phi)
    vz = cos(2pi * beta3 * (z - 1) + phi)

    return 2sqrt(2) * vx * vy * vz
end

function onsite_unit_cell_value(i)
    x, y, z = xyz_from_linear_index(i)

    # Coordinates within the 4-site modulation cell: 0, 1, 2, 3.
    ax = mod(x - 1, 4)
    ay = mod(y - 1, 4)
    az = mod(z - 1, 4)

    # Unit-cell coordinates: 1, 2, 3, ...
    cx = div(x - 1, 4) + 1
    cy = div(y - 1, 4) + 1
    cz = div(z - 1, 4) + 1

    vx = cos(2pi * beta1 * (cx - 1) + phi)
    vy = cos(2pi * beta2 * (cy - 1) + phi)
    vz = cos(2pi * beta3 * (cz - 1) + phi)

    return 2sqrt(2) * lambda_strength * vx * vy * vz
end


onsite_unit_cell_value (generic function with 1 method)

## Build the tensor-network Hamiltonian

The `xy_mpo` contains intra-row and inter-row hopping. It is embedded into the full 3D coordinate register, then combined with vertical hopping and the imaginary onsite loss term.


In [29]:
site_xy = siteinds("Qubit", 2L; conserve_qns=false)
site_z = siteinds("Qubit", L; conserve_qns=false)
sites = vcat(site_z, site_xy)

id_xy = identity_mpo(site_xy, 2L)
id_z = identity_mpo(site_z, L)
id_all = identity_mpo(sites, 3L)

hop_x = alternating_row_hopping(2L, tx1, tx2, site_xy)
hop_y = staggered_column_hopping(2L, tx1, tx2, site_xy)

intra_mpo = intrachain_hopping(2^L, hop_x, 2^(2L), site_xy, id_xy)
inter_mpo = interchain_hopping_square(2^L, hop_y, 2^(2L), site_xy)
xy_mpo = intra_mpo + inter_mpo
xy_mpo_embedded, _ = concatenate_MPOs(id_z, site_z, xy_mpo, site_xy)

hop_z = vertical_hopping(3L, tx1, tx2, sites)
z_mpo = interchain_hopping_square(2^(2L), hop_z, 2^(3L), sites)


MPO
[1] ((dim=2|id=31|"Qubit,Site,n=1")', (dim=2|id=31|"Qubit,Site,n=1"), (dim=4|id=246|"l=1,link"))
[2] ((dim=2|id=982|"Qubit,Site,n=2")', (dim=2|id=982|"Qubit,Site,n=2"), (dim=4|id=619|"l=2,link"), (dim=4|id=246|"l=1,link"))
[3] ((dim=2|id=970|"Qubit,Site,n=3")', (dim=2|id=970|"Qubit,Site,n=3"), (dim=2|id=529|"l=3,link"), (dim=4|id=619|"l=2,link"))
[4] ((dim=2|id=444|"Qubit,Site,n=4")', (dim=2|id=444|"Qubit,Site,n=4"), (dim=1|id=594|"l=4,link"), (dim=2|id=529|"l=3,link"))
[5] ((dim=2|id=416|"Qubit,Site,n=1")', (dim=2|id=416|"Qubit,Site,n=1"), (dim=1|id=582|"l=5,link"), (dim=1|id=594|"l=4,link"))
[6] ((dim=2|id=515|"Qubit,Site,n=2")', (dim=2|id=515|"Qubit,Site,n=2"), (dim=1|id=797|"l=6,link"), (dim=1|id=582|"l=5,link"))
[7] ((dim=2|id=305|"Qubit,Site,n=3")', (dim=2|id=305|"Qubit,Site,n=3"), (dim=1|id=868|"l=7,link"), (dim=1|id=797|"l=6,link"))
[8] ((dim=2|id=232|"Qubit,Site,n=4")', (dim=2|id=232|"Qubit,Site,n=4"), (dim=1|id=125|"l=8,link"), (dim=1|id=868|"l=7,link"))
[9] ((dim=2|id=68

In [30]:
xvals = range(1, 2^(3L); length=2^(3L))

qtt_cell, _, _ = quanticscrossinterpolate(Float64, onsite_unit_cell_value, xvals; tolerance=1e-8)
onsite_cell_mps = MPS(TCI.tensortrain(qtt_cell.tci); sites)

qtt_env, _, _ = quanticscrossinterpolate(Float64, onsite_envelope, xvals; tolerance=1e-8)
onsite_envelope_mps = MPS(TCI.tensortrain(qtt_env.tci); sites)

loss_mpo = 1im * apply(
    mps_to_diagonal_mpo(onsite_cell_mps, sites),
    mps_to_diagonal_mpo(onsite_envelope_mps, sites),
)

tot_mpo = xy_mpo_embedded + z_mpo + loss_mpo


MPO
[1] ((dim=2|id=31|"Qubit,Site,n=1")', (dim=2|id=31|"Qubit,Site,n=1"), (dim=5|id=142|"Link,l=1"))
[2] ((dim=2|id=982|"Qubit,Site,n=2")', (dim=2|id=982|"Qubit,Site,n=2"), (dim=5|id=151|"Link,l=2"), (dim=5|id=142|"Link,l=1"))
[3] ((dim=2|id=970|"Qubit,Site,n=3")', (dim=2|id=970|"Qubit,Site,n=3"), (dim=4|id=61|"Link,l=3"), (dim=5|id=151|"Link,l=2"))
[4] ((dim=2|id=444|"Qubit,Site,n=4")', (dim=2|id=444|"Qubit,Site,n=4"), (dim=3|id=59|"Link,l=4"), (dim=4|id=61|"Link,l=3"))
[5] ((dim=2|id=416|"Qubit,Site,n=1")', (dim=2|id=416|"Qubit,Site,n=1"), (dim=5|id=98|"Link,l=1"), (dim=3|id=59|"Link,l=4"))
[6] ((dim=2|id=515|"Qubit,Site,n=2")', (dim=2|id=515|"Qubit,Site,n=2"), (dim=5|id=933|"Link,l=2"), (dim=5|id=98|"Link,l=1"))
[7] ((dim=2|id=305|"Qubit,Site,n=3")', (dim=2|id=305|"Qubit,Site,n=3"), (dim=5|id=466|"Link,l=3"), (dim=5|id=933|"Link,l=2"))
[8] ((dim=2|id=232|"Qubit,Site,n=4")', (dim=2|id=232|"Qubit,Site,n=4"), (dim=3|id=575|"Link,l=4"), (dim=5|id=466|"Link,l=3"))
[9] ((dim=2|id=68|"Qubi

## Hermitize `omega * I - H`

For a non-Hermitian Hamiltonian, the spectral function is evaluated through the Hermitized block operator. The auxiliary qubit distinguishes the upper and lower blocks.


In [31]:
omega = kx + 1im * ky
upper_block = omega * id_all - tot_mpo
lower_block = dag(upper_block)

aux_site = siteinds("Qubit", 1; conserve_qns=false)

up_ops = OpSum()
up_ops += 1, "sigma_plus", 1
up_mpo = MPO(up_ops, aux_site)

down_ops = OpSum()
down_ops += 1, "sigma_minus", 1
down_mpo = MPO(down_ops, aux_site)

ham_u, hermitized_sites = concatenate_MPOs(up_mpo, aux_site, upper_block, sites)
ham_d, _ = concatenate_MPOs(down_mpo, aux_site, lower_block, sites)
hermitized_ham = ham_u + ham_d

I_ldn, _ = concatenate_MPOs(down_mpo, aux_site, id_all, sites)


(MPO
[1] ((dim=2|id=17|"Qubit,Site,n=1")', (dim=2|id=17|"Qubit,Site,n=1"), (dim=1|id=529|"Link,l=1"))
[2] ((dim=2|id=917|"Link,l=1"), (dim=2|id=31|"Qubit,Site,n=1")', (dim=2|id=31|"Qubit,Site,n=1"), (dim=1|id=529|"Link,l=1"))
[3] ((dim=2|id=917|"Link,l=1"), (dim=2|id=662|"Link,l=2"), (dim=2|id=982|"Qubit,Site,n=2")', (dim=2|id=982|"Qubit,Site,n=2"))
[4] ((dim=2|id=662|"Link,l=2"), (dim=2|id=485|"Link,l=3"), (dim=2|id=970|"Qubit,Site,n=3")', (dim=2|id=970|"Qubit,Site,n=3"))
[5] ((dim=2|id=485|"Link,l=3"), (dim=2|id=205|"Link,l=4"), (dim=2|id=444|"Qubit,Site,n=4")', (dim=2|id=444|"Qubit,Site,n=4"))
[6] ((dim=2|id=205|"Link,l=4"), (dim=2|id=839|"Link,l=5"), (dim=2|id=416|"Qubit,Site,n=1")', (dim=2|id=416|"Qubit,Site,n=1"))
[7] ((dim=2|id=839|"Link,l=5"), (dim=2|id=976|"Link,l=6"), (dim=2|id=515|"Qubit,Site,n=2")', (dim=2|id=515|"Qubit,Site,n=2"))
[8] ((dim=2|id=976|"Link,l=6"), (dim=2|id=132|"Link,l=7"), (dim=2|id=305|"Qubit,Site,n=3")', (dim=2|id=305|"Qubit,Site,n=3"))
[9] ((dim=2|id=132

## KPM evaluation at one site

This cell computes the Chebyshev recurrence and contracts the Jackson-weighted series into a single local DOS value.


In [32]:
moments = KPM_Tn_NH_bysite(
    hermitized_ham,
    kpm_order,
    site_index,
    spectrum_scale,
    hermitized_sites,
    I_ldn,
)

dos_point = get_energy_from_T_MPS(moments, kpm_order, site_index, hermitized_sites)
println("DOS at omega = $(omega): $(dos_point)")


DOS at omega = 0.0 + 4.7im: 382.060650313423 + 3.146895869636006e-11im
